In [4]:
import numpy as np

BASE = r"D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)"
DIR  = BASE + r"\data\mean_pooled_scaled"

splits = {}
for name in ["train", "val", "test"]:
    d = np.load(f"{DIR}\\ppi_{name}_mean_pooled_scaled.npz", allow_pickle=True)
    splits[name] = d
    print(f"\n===== {name} =====")
    print("files:", d.files)
    for k in d.files:
        a = d[k]
        print(f"  {k}: shape={a.shape}, dtype={a.dtype}")
        # 标签数组：值少于 5 种就打印分布
        if a.ndim == 1 and a.dtype.kind in "iub" and len(np.unique(a)) <= 5:
            u, c = np.unique(a, return_counts=True)
            print(f"    标签分布: {dict(zip(u.tolist(), c.tolist()))}")
        # ID 数组：打印样例
        if a.dtype.kind in "US" or a.dtype == object:
            print(f"    样例: {a[:3]}")

# ---- 缩放来源指纹（关键检查）----
def fingerprint(X, tag):
    mu, sd = X.mean(0), X.std(0)
    print(f"[{tag}] N={len(X)}, dims={X.shape[1]}")
    print(f"  per-dim mean : max|μ|={np.abs(mu).max():.3e}")
    print(f"  per-dim std  : min={sd.min():.4f}, mean={sd.mean():.4f}, max={sd.max():.4f}")
    print(f"  value range  : [{X.min():.2f}, {X.max():.2f}]")

Xtr = splits["train"]["X"]          # 键名不同就换成 d.files 里的实际名字
fingerprint(Xtr, "train")
fingerprint(splits["val"]["X"], "val")
fingerprint(splits["test"]["X"], "test")

# train-fit 正确时，val 的 per-dim mean 应有 ≈1/√N 的自然波动
print("\n参考: 1/√N_val =", 1/np.sqrt(len(splits['val']['X'])),
      "| val μ 跨维波动 =", splits['val']['X'].mean(0).std())



===== train =====
files: ['X', 'y']
  X: shape=(163192, 2560), dtype=float32
  y: shape=(163192,), dtype=int64
    标签分布: {0: 81596, 1: 81596}

===== val =====
files: ['X', 'y']
  X: shape=(59260, 2560), dtype=float32
  y: shape=(59260,), dtype=int64
    标签分布: {0: 29630, 1: 29630}

===== test =====
files: ['X', 'y']
  X: shape=(52048, 2560), dtype=float32
  y: shape=(52048,), dtype=int64
    标签分布: {0: 26024, 1: 26024}
[train] N=163192, dims=2560
  per-dim mean : max|μ|=5.140e-07
  per-dim std  : min=0.9999, mean=1.0000, max=1.0000
  value range  : [-9.80, 11.20]
[val] N=59260, dims=2560
  per-dim mean : max|μ|=7.957e-01
  per-dim std  : min=0.8380, mean=1.0881, max=1.4380
  value range  : [-11.06, 9.86]
[test] N=52048, dims=2560
  per-dim mean : max|μ|=7.901e-01
  per-dim std  : min=0.8761, mean=1.0999, max=1.4365
  value range  : [-10.63, 9.65]

参考: 1/√N_val = 0.004107893507034559 | val μ 跨维波动 = 0.23624794


In [5]:
import pickle, numpy as np, pandas as pd

BASE = r"D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)"
with open(BASE + r"\Embeddings\embeddings_mean.pkl", "rb") as f:
    emb = pickle.load(f)
E = {k: np.asarray(v, np.float32).reshape(-1) for k, v in emb.items()}   # 顺手修掉 (1,1280)

def load_pairs(split, label):
    df = pd.read_csv(f"{BASE}\\dataset\\{split}_{'pos' if label else 'neg'}_rr.txt",
                     sep=r"\s+", header=None, names=["a","b"]).dropna()
    miss = ~(df["a"].isin(E) & df["b"].isin(E))
    return df[~miss], int(miss.sum())

def build(df):
    return np.hstack([np.stack([E[a] for a in df["a"]]),
                      np.stack([E[b] for b in df["b"]])])

tr = pd.concat([load_pairs("Intra1",1)[0], load_pairs("Intra1",0)[0]])
print("Intra1 缺嵌入蛋白对:", load_pairs("Intra1",1)[1] + load_pairs("Intra1",0)[1])  # 覆盖率门禁

mu, sd = build(tr).mean(0), build(tr).std(0)                # train-fit，与原协议对齐
va = pd.concat([load_pairs("Intra0",1)[0], load_pairs("Intra0",0)[0]])
mu_re = ((build(va) - mu) / sd).mean(0)

old = np.load(BASE + r"\data\mean_pooled_scaled\ppi_val_mean_pooled_scaled.npz")["X"].mean(0)
r1 = np.corrcoef(mu_re, old)[0,1]                           # [h1;h2] 顺序
r2 = np.corrcoef(mu_re[1280:], old[:1280].tolist() + [] )   # 防 [h2;h1] 顺序：见下
r2 = np.corrcoef(np.concatenate([mu_re[1280:], mu_re[:1280]]), old)[0,1]
print(f"指纹相关 r(同序)={r1:.4f}, r(换序)={r2:.4f}, 逐维最大偏差={np.abs(mu_re-old).max():.4f}")


Intra1 缺嵌入蛋白对: 0
指纹相关 r(同序)=0.9999, r(换序)=0.9821, 逐维最大偏差=0.0098
